In [ ]:
import arcpy
import os
arcpy.env.overwriteOutput = True

#----------------------------------------
# CONFIGURATION - Set your local paths here
#----------------------------------------
folder = r'C:\Users\AJ\Desktop\PY'
residential = os.path.join(folder , 'Residential.shp')
schools = os.path.join(folder , 'School.shp')
sorted_school = os.path.join(folder , 'sorted_school.shp')
population_field = "population"
capacity_field = "capacity"

#----------------------------------------
service_id_field = "Service_ID"
area_field = "Area_Field"
print("Checking fields...")

#----------------------------------------
# checking existing fields
#----------------------------------------
print("checking your fields")
res_fields = [field.name for field in arcpy.ListFields(residential)]
school_fields = [field.name for field in arcpy.ListFields(schools)]

if capacity_field not in school_fields:
    print(f"error: capacity field couldn't be found in your fields")
    raise SystemExit
else:
    print(f'{capacity_field} has been successfully found')

if population_field not in res_fields:
    print(f"error: population field couldn't be found in your residential fields")
    raise SystemExit
else:
    print(f'{population_field} has been successfully found')

#----------------------------------------
# making necessary fields
#----------------------------------------
print('preparing new vital fields')
if service_id_field in res_fields:
    arcpy.management.DeleteField(residential, service_id_field, 'DELETE_FIELDS')
    arcpy.management.AddField(residential, service_id_field, "LONG")
    print('Service ID field was deleted and then added to the residential layer')
else:
    arcpy.management.AddField(residential, service_id_field, "LONG")
    print('Service ID field was added to the residential layer')

if area_field in school_fields:
    print("Area field already exists")
    arcpy.management.DeleteField(schools, area_field, 'DELETE_FIELDS')
    print("Area field has been deleted")
    arcpy.management.AddField(schools, area_field, "DOUBLE")
    print('New area field has been added')
else:
    arcpy.management.AddField(schools, area_field, "DOUBLE")
    print('area field has been added to the school layer')

if service_id_field in school_fields:
    print("Service_ID field already exist")
    arcpy.management.DeleteField(schools, service_id_field, 'DELETE_FIELDS')
    print("Service_ID field has been deleted")
    arcpy.management.AddField(schools, service_id_field, "LONG")
    print("New Service_ID field has been added")
else:
    arcpy.management.AddField(schools, service_id_field, "LONG")
    print('Service ID field has been added to the school layer')

#----------------------------------------
# Updating field lists
#----------------------------------------
res_fields = [field.name for field in arcpy.ListFields(residential)]
school_fields = [field.name for field in arcpy.ListFields(schools)]

#----------------------------------------
# resetting Residential Service_ID to 0
#----------------------------------------
print("Resetting residential Service_ID to 0...")
with arcpy.da.UpdateCursor(residential, [service_id_field]) as cu:
    for row in cu:
        row[0] = 0
        cu.updateRow(row)
print('the Service ID Fields was reset to 0')

#----------------------------------------
# Calculating Area Feild
#----------------------------------------
print('calculating Area for school')
arcpy.management.CalculateGeometryAttributes(schools, [[area_field, "AREA"]], area_unit="SQUARE_METERS")

#----------------------------------------
# Sorting Area by capacity and Shape_Area
#----------------------------------------
print('sorting school fields')
sort_criteria = [[capacity_field, "DESCENDING"], [area_field, "DESCENDING"]]
try:
    arcpy.management.Sort(schools, sorted_school, sort_criteria)
    print('sorting operation was done and now, it is saved into sorted_school shapefile')
except Exception as e:
    print(f'error {e} occured')
    raise SystemExit

#----------------------------------------
# allocating numbers to sorted schools
#----------------------------------------
print('allocating numbers was started')
countit = 1

with arcpy.da.UpdateCursor(sorted_school, [service_id_field]) as cu:
    for row in cu:
        row[0] = countit
        countit += 1
        cu.updateRow(row)
print('allocating numbers was finished successfully')

#----------------------------------------
# making feature layers
#----------------------------------------
arcpy.management.MakeFeatureLayer(residential, 'Residential_Layer')
arcpy.management.MakeFeatureLayer(sorted_school, 'School_Layer')
print("Feature layers created.")

#----------------------------------------
# SQL delimiters for compatibility
#----------------------------------------
school_sql = arcpy.AddFieldDelimiters("School_Layer", service_id_field)
res_sql = arcpy.AddFieldDelimiters("Residential_Layer", service_id_field)

#----------------------------------------
# default parameters
#----------------------------------------
fields_to_read = [service_id_field, capacity_field]
min_radius_start = 0
max_radius_start = 3500
max_iterations = 10

#----------------------------------------
# Clearing previous selections
#----------------------------------------
print("The main action has just been started; it may take some minutes accroding to the numbers of your schools")
arcpy.management.SelectLayerByAttribute('School_Layer', "CLEAR_SELECTION")
arcpy.management.SelectLayerByAttribute('Residential_Layer', "CLEAR_SELECTION")

#----------------------------------------
# counting schools
#----------------------------------------
count = int(arcpy.management.GetCount('School_Layer')[0])
print(f'Total schools = {count}')

#----------------------------------------
# looping schools
#----------------------------------------
with arcpy.da.SearchCursor('School_Layer', fields_to_read) as school_cursor:
    for s_row in school_cursor:

        current_ID = s_row[0]
        current_capacity = s_row[1]

        if current_capacity is None or current_capacity <= 0:
            print(f"Warning: school {current_ID} has None or invalid capacity. Skipping...")
            continue

        print(f'school = {current_ID}')
        print(f'capacity = {current_capacity}')

        min_radius = min_radius_start
        max_radius = max_radius_start
        best_radius = None
        best_population = None
        best_difference = None

        for i in range(max_iterations):

            print(f'Iteration {i+1}')

            arcpy.management.SelectLayerByAttribute(
                'School_Layer',
                "NEW_SELECTION",
                f"{school_sql} = {current_ID}"
            )

            test_radius = (max_radius + min_radius) / 2

            print(f'max Radius = {max_radius}')
            print(f'min Radius = {min_radius}')
            print(f'test radius = {test_radius}')

            temp_buffer = os.path.join("in_memory", 'temp_buffer')

            if arcpy.Exists(temp_buffer):
                arcpy.management.Delete(temp_buffer)

            arcpy.analysis.Buffer('School_Layer', temp_buffer, f'{test_radius} Meters')

            arcpy.management.SelectLayerByAttribute(
                'Residential_Layer',
                "NEW_SELECTION",
                f"{res_sql} = 0"
            )

            arcpy.management.SelectLayerByLocation(
                'Residential_Layer',
                "HAVE_THEIR_CENTER_IN",
                temp_buffer,
                selection_type="SUBSET_SELECTION"
            )

            population = 0

            with arcpy.da.SearchCursor('Residential_Layer', [population_field]) as pop_cursor:
                for pop_row in pop_cursor:
                    if pop_row[0] is not None:
                        population += pop_row[0]

            print(f'population covered by iteration {i+1} is equal to {population}')

            difference = abs(population - current_capacity)

            if best_difference is None or difference < best_difference:
                best_difference = difference
                best_population = population
                best_radius = test_radius

            if population <= current_capacity:
                min_radius = test_radius
            else:
                max_radius = test_radius

            if arcpy.Exists(temp_buffer):
                arcpy.management.Delete(temp_buffer)

        print(f'cursor reading school {current_ID}')
        print(f'in the best: radius is {best_radius} - it covers {best_population} people with {best_difference} difference in general Iteration')

        if best_radius is not None:

            final_buffer = os.path.join("in_memory", 'final_buffer')

            if arcpy.Exists(final_buffer):
                arcpy.management.Delete(final_buffer)

            arcpy.management.SelectLayerByAttribute(
                'School_Layer',
                "NEW_SELECTION",
                f"{school_sql} = {current_ID}"
            )

            arcpy.analysis.Buffer("School_Layer", final_buffer, f'{best_radius} Meters')

            arcpy.management.SelectLayerByAttribute(
                "Residential_Layer",
                "NEW_SELECTION",
                f"{res_sql} = 0"
            )

            arcpy.management.SelectLayerByLocation(
                'Residential_Layer',
                "HAVE_THEIR_CENTER_IN",
                final_buffer,
                selection_type="SUBSET_SELECTION"
            )

            selected_count = int(arcpy.management.GetCount('Residential_Layer')[0])
            print(f'Residential selected for final assignment = {selected_count}')

            with arcpy.da.UpdateCursor("Residential_Layer", [service_id_field]) as final_cursor:
                for final_row in final_cursor:
                    final_row[0] = current_ID
                    final_cursor.updateRow(final_row)

            print(f'assignment done for school {current_ID}')

            arcpy.management.Delete(final_buffer)


Checking fields...
checking your fields
capacity has been successfully found
population has been successfully found
preparing new vital fields
Service ID field was added to the residential layer
area field has been added to the school layer
Service ID field has been added to the school layer
Resetting residential Service_ID to 0...
the Service ID Fields was reseted to 0
calculating Area for school
sorting school fields
sorting operation was done and now, it is saved into sorted_school shapefile
allocating numbers was started
allocating numbers was finished successfully
Feature layers created.
The main action has just been started; it may take some minutes accroding to the numbers of your schools
Total schools = 183
school = 1
capacity = 2983
Iteration 1
max Radius = 3500
min Radius = 0
test radius = 1750.0
population covered by iteration 1 is equal to 77341
Iteration 2
max Radius = 1750.0
min Radius = 0
test radius = 875.0
population covered by iteration 2 is equal to 20438
Iteration 3